# Recording Audio

In [[807A26DA-D106-4E6B-9921-8B6BCC8054E2|Playing Notes and Sounds]] your code made sound. In this notebook it listens. `audio.record` captures sound from your **microphone** and saves it as a sound file that the rest of your code can play, edit, measure, or send somewhere else.

Here is what happens when a cell records:

1. The first time, your browser asks whether this page may use the microphone. Choose **Allow**.
2. A small recorder appears under the cell, with a timer and a **level meter** that jumps when it hears you.
3. When the recording ends, the sound is saved to the file you named, and `audio.record` gives back the path to that file.

Run the cell below and say something. It records for three seconds, then plays your voice back.

> **Good to know:** A web page can never listen without asking first. If you chose *Block* by mistake, click the icon at the left of the address bar to change the microphone setting, then run the cell again. The recording stays on your computer. Nothing is uploaded unless your own code sends it.

In [ ]:
from codetto import audio

path = audio.record('/tmp/hello.wav', seconds=3)
print('Saved to', path)

audio.play(path)

# Recording Until You Stop

`seconds=3` made a **timed** recording. Leave `seconds` out and the recording is **open-ended**: it keeps going until you click **Stop recording** on the recorder.

Timed recordings suit code that needs clips of the same length, such as collecting examples to compare. Open-ended recordings suit a voice note, where you don't know how long you will talk.

Run the cell, talk for as long as you like, then click **Stop recording**.

> **Tip:** The **Stop recording** button finishes the clip and lets the rest of the cell carry on. The cell's own Stop button (■) is different. It cancels the whole cell, and nothing is saved.

In [ ]:
path = audio.record('/tmp/voice_note.wav')

print('Finished recording.')
audio.play(path)

# Choosing a Microphone

Many computers have more than one microphone: one built in, one on a headset, one on a webcam. `audio.record` uses the default one unless you say otherwise.

`audio.list_microphones()` returns a list of the microphones your browser can see. Each one is a dictionary with a `name` and an `id`.

To record from a particular microphone, pass it as `device`. You can give one of the items from the list, or just part of its name, such as `device='Headset'`.

Change the `0` in the cell below to try a different microphone from your list.

In [ ]:
mics = audio.list_microphones()

for number, mic in enumerate(mics):
  print(number, mic['name'])

path = audio.record('/tmp/mic_test.wav', seconds=2, device=mics[0])
audio.play(path)

# Keeping a Recording

Where you save a recording decides how long it lasts.

| Path starts with | What happens to the file |
|---|---|
| `/tmp/` | Kept only while this page is open. It is gone after a reload. |
| `/attachments/` | Saved inside the notebook as an **attachment**, so it is still there next time. |

A recording saved under `/attachments/` shows up in the notebook's attachments list (the paperclip icon), where you can download it as a normal `.wav` file. The [[81424A59-6B7C-4A5A-8832-B13A6B9CBB13|Attachments]] guide explains more.

Each attachment can be up to 5 MB. A recording uses about 32 KB for every second, so that is room for about two and a half minutes. Use `/tmp/` for quick experiments and `/attachments/` for clips you want to keep.

The cell below records a clip to keep. The rest of this notebook uses it, so say a full sentence.

In [ ]:
import os

path = audio.record('/attachments/my_voice.wav', seconds=4)

size = os.path.getsize(path)
print(f'{path} is {size / 1024:.0f} KB')

# Sound as Numbers

Sound is air pressure moving up and down very quickly. A microphone measures that pressure thousands of times every second, and each measurement is called a **sample**. The number of samples taken each second is the **sample rate**.

`audio.load(path)` opens a recording and gives you both:

- `samples` is a NumPy array with one number per sample. Every number is between `-1` and `1`. Silence is close to `0`, and loud sounds swing further from it.
- `rate` is the sample rate. Codetto records at 16,000 samples per second unless you ask for something else.

Because a recording is just an array, everything from [[A6AFAE39-B14C-4CD8-B9CB-23771BBF07C9|NumPy]] works on it.

In [ ]:
import numpy as np

samples, rate = audio.load('/attachments/my_voice.wav')

print('Sample rate:', rate, 'samples per second')
print('Samples:', len(samples))
print('Length:', round(len(samples) / rate, 2), 'seconds')
print('Loudest sample:', round(float(np.max(np.abs(samples))), 3))
print('Ten samples from the middle:', samples[len(samples) // 2:][:10].round(3))

## Seeing the Sound

Plot the samples in order and you get a **waveform**, a picture of the recording. Tall parts are loud and flat parts are quiet, so you can see each word you said.

The x-axis needs the time of every sample. Sample number 16,000 happens one second in, so dividing each sample's position by the rate gives its time in seconds. There is more about charts in [[F9D25507-BA02-4876-9F0C-CD35B389BC47|Matplotlib]].

In [ ]:
import matplotlib.pyplot as plt

times = np.arange(len(samples)) / rate

plt.figure(figsize=(8, 3))
plt.plot(times, samples, linewidth=0.5)
plt.ylim(-1, 1)
plt.xlabel('Time (seconds)')
plt.ylabel('Level')
plt.title('My recording')
plt.show()

# Editing a Recording

Editing sound is array maths. The cell below makes two common fixes:

- **Trimming** removes the silence before you started talking and after you finished. It finds every sample louder than a small threshold, then keeps the slice from the first loud one to the last.
- **Normalizing** makes a quiet recording louder. It divides by the loudest sample, so the loudest point lands just under `1`.

`audio.save(path, samples, rate)` writes an array back out as a sound file that `audio.play` can play.

In [ ]:
loud = np.flatnonzero(np.abs(samples) > 0.02)

if len(loud) == 0:
  print('That recording is silent. Record it again, a little louder.')
  clean = samples
else:
  clean = samples[loud[0]:loud[-1] + 1]
  clean = clean / np.max(np.abs(clean)) * 0.9

audio.save('/tmp/clean.wav', clean, rate)
print(f'Trimmed {len(samples) / rate:.1f} seconds down to {len(clean) / rate:.1f} seconds')
audio.play('/tmp/clean.wav')

## Change the Speed

The sample rate tells the player how quickly to read through the samples. Save the *same* samples with a *higher* rate and they are played faster, which also makes your voice higher. A lower rate makes it slow and deep.

Reversing the array with `[::-1]` plays the recording backwards.

Drag the slider or flip the switch, and the clip plays again with your changes.

In [ ]:
SPEED = 1.5 #@param {type:"slider", min:0.5, max:2, step:0.1}
BACKWARDS = False #@param {type:"boolean"}

remix = clean[::-1] if BACKWARDS else clean

audio.save('/tmp/remix.wav', remix, int(rate * SPEED))
audio.play('/tmp/remix.wav')

# Measuring a Recording

Your code can also *measure* a sound. One useful measurement is its **pitch**: how high or low it is, counted in vibrations per second, or **hertz** (Hz).

NumPy's `np.fft.rfft` takes a recording apart into the pitches it is made of, and tells you how strong each one is. The strongest one is the note you hear.

Run the cell, then **whistle or hum one steady note** for two seconds. The code finds the strongest pitch, works out the nearest musical note, and plays that note back so you can compare. The note formula is the one from [[807A26DA-D106-4E6B-9921-8B6BCC8054E2|Playing Notes and Sounds]] used in reverse, where `A4` is 440 Hz.

In [ ]:
import math

path = audio.record('/tmp/pitch.wav', seconds=2)
tone, tone_rate = audio.load(path)

strength = np.abs(np.fft.rfft(tone))
pitches = np.fft.rfftfreq(len(tone), 1 / tone_rate)
strength[pitches < 80] = 0  # ignore low rumble

pitch = pitches[np.argmax(strength)]
print(f'Strongest pitch: {pitch:.0f} Hz')

if pitch > 0:
  names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
  steps = round(12 * math.log2(pitch / 440)) + 57  # A4 is step 57
  note = names[steps % 12] + str(steps // 12)
  print('Nearest note:', note)
  audio.play_note(note, 1)

# Sending a Recording to an AI Model

A recording is a file, so you can send it to an online service. Some AI models accept sound as well as text. Give one your recording and ask it to write down what was said, and you have **speech-to-text**.

This cell needs an `OPENROUTER_API_KEY` environment variable. [[503B5721-E671-4BE2-992C-2B8782BC43F8|Environment Variables]] shows how to add one, and [[26056657-3785-482A-8408-C85C447D8D25|Using the OpenAI SDK]] explains the code that talks to the model.

The file's bytes are turned into text with `base64` so they can travel inside the message. The message then has two parts, the instruction and the audio.

> **Good to know:** This is the one cell in the notebook that sends your recording away from your computer.

In [ ]:
import base64
import openai
import os

client = openai.OpenAI(
  base_url='https://openrouter.ai/api/v1',
  api_key=os.environ['OPENROUTER_API_KEY']
)

MODEL = "google/gemini-2.5-flash" #@param ["google/gemini-2.5-flash", "openai/gpt-4o-audio-preview", "mistralai/voxtral-small-24b-2507"]

with open('/attachments/my_voice.wav', 'rb') as f:
  audio_text = base64.b64encode(f.read()).decode()

response = client.chat.completions.create(
  model=MODEL,
  messages=[{
    "role": "user",
    "content": [
      {"type": "text", "text": "Transcribe this recording exactly. Reply with only the transcript."},
      {"type": "input_audio", "input_audio": {"data": audio_text, "format": "wav"}},
    ],
  }]
)

print(response.choices[0].message.content)

# What Else Can You Build?

Recording, loading, and saving are the building blocks. Here are some things to make with them:

- **A voice memo notebook.** Save a spoken note under `/attachments/` with a different file name each time.
- **A clap switch.** Record one second at a time in a loop, and do something when the loudest sample goes over a threshold.
- **A tuner.** Put the pitch cell in a loop and show whether each note is too high or too low.
- **A sound effects studio.** Record your own effects, trim and normalize them, then play them from a game in [[699987CF-F87D-4E8C-9F64-C8DCBB6DC0FF|Pygame]].
- **A sound classifier.** Record ten claps and ten whistles, measure the pitch and loudness of each, and train a model to tell them apart.
- **A talking assistant.** Transcribe a spoken question, send the text to an AI model, and read the answer aloud with [[F3C9A21D-6B4E-4D87-9A0C-5E1F2A3B4C6D|Text to Speech]].

# Check Your Understanding